In [4]:
import time
import csv
from datetime import datetime, timedelta
from pymongo import MongoClient
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By

# ========== MONGODB (CORREGIDO) ==========
try:
    client = MongoClient("mongodb://database:27017/")  # ← CAMBIADO
    db = client["proyecto_bigdata"]
    coleccion = db["alojamientos"]
    print("✅ MongoDB conectado")
except Exception as e:
    print(f"❌ Error MongoDB: {e}")
    exit()

# ========== CONFIGURACIÓN SELENIUM ==========
options = Options()
options.binary_location = "/usr/bin/google-chrome"
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1920,1080")

service = Service("/usr/bin/chromedriver")
driver = webdriver.Chrome(service=service, options=options)

# ========== FUNCIONES ==========
def limpiar_precio(texto):
    numeros = ''.join(c for c in texto if c.isdigit())
    return int(numeros) if numeros else None

def determinar_zona(ciudad):
    norte = ['Arica','Iquique','Calama','Antofagasta','Copiapo','La Serena']
    centro = ['Valparaiso','Vina del Mar','Santiago','Rancagua','Talca','Chillan','Concepcion']
    
    if ciudad in norte:
        return "Norte"
    elif ciudad in centro:
        return "Centro"
    else:
        return "Sur"

# ========== CIUDADES ==========
ciudades = [
    "Arica","Iquique","Calama","Antofagasta","Copiapo",
    "La-Serena","Valparaiso","Vina-del-Mar","Santiago","Rancagua",
    "Talca","Chillan","Concepcion","Temuco","Valdivia",
    "Puerto-Varas","Puerto-Montt","Coyhaique","Puerto-Natales","Punta-Arenas"
]

checkin = (datetime.now() + timedelta(days=1)).strftime("%Y-%m-%d")
checkout = (datetime.now() + timedelta(days=4)).strftime("%Y-%m-%d")

print(f"📅 Fechas: {checkin} → {checkout}")

total_guardados = 0

# ========== SCRAPING ==========
for idx, ciudad in enumerate(ciudades):
    ciudad_limpia = ciudad.replace("-", " ")

    print(f"\n📍 [{idx+1}/{len(ciudades)}] {ciudad_limpia}")

    url = f"https://www.kayak.cl/hotels/{ciudad}/{checkin}/{checkout}/2adults"
    driver.get(url)
    time.sleep(5)

    try:
        driver.find_element(By.CSS_SELECTOR, "button[aria-label='Cerrar']").click()
        time.sleep(1)
    except:
        pass

    for _ in range(4):
        driver.execute_script("window.scrollBy(0, 800);")
        time.sleep(1.5)

    precios_elements = driver.find_elements(By.CSS_SELECTOR, "div[data-target='price']")
    nombres_elements = driver.find_elements(By.CSS_SELECTOR, "a.c9Hnq-big-name")

    print(f"   🔎 Nombres: {len(nombres_elements)} | Precios: {len(precios_elements)}")

    guardados_ciudad = 0

    for i in range(min(len(nombres_elements), len(precios_elements))):
        try:
            nombre = nombres_elements[i].text.strip()
            precio_texto = precios_elements[i].text.strip()
            precio = limpiar_precio(precio_texto)

            if not nombre:
                continue

            zona = determinar_zona(ciudad_limpia)

            registro = {
                "nombre_hotel": nombre,
                "precio_noche": precio,
                "precio_texto": precio_texto,
                "ciudad": ciudad_limpia,
                "zona_geografica": zona,
                "checkin": checkin,
                "checkout": checkout,
                "fecha_captura": datetime.now(),
                "plataforma": "Kayak.cl",
                "integrante": "Lucas-Cheuque",
                "grupo": "G5_Turismo_Hoteleria"
            }

            coleccion.update_one(
                {
                    "nombre_hotel": nombre,
                    "ciudad": ciudad_limpia,
                    "plataforma": "Kayak.cl"
                },
                {"$set": registro},
                upsert=True
            )

            guardados_ciudad += 1
            print(f"   ✅ Guardado: {nombre[:40]}")

        except Exception as e:
            print(f"   ⚠️ Error: {e}")

    total_guardados += guardados_ciudad
    print(f"   📊 Guardados en {ciudad_limpia}: {guardados_ciudad}")

    time.sleep(5)

driver.quit()

# ========== BACKUP CSV ==========
datos = list(coleccion.find({"plataforma": "Kayak.cl"}, {"_id": 0}))

if datos:
    with open("kayak_final_mongodb.csv", "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=datos[0].keys())
        writer.writeheader()
        writer.writerows(datos)
    print(f"\n💾 CSV generado con {len(datos)} registros")

print(f"\n🎉 TOTAL GUARDADOS: {total_guardados}")

✅ MongoDB conectado
📅 Fechas: 2026-04-23 → 2026-04-26

📍 [1/20] Arica
   🔎 Nombres: 26 | Precios: 26
   ✅ Guardado: Hotel Plaza Colon
   ✅ Guardado: Hotel Gavina Express Arica
   ✅ Guardado: Hotel Y Restaurant Isidora
   ✅ Guardado: Hotel Andalucía
   ✅ Guardado: Panamericana Hotel Arica
   ✅ Guardado: Hotel Concorde
   ✅ Guardado: Hotel Samaña
   ✅ Guardado: Amaru Hotel
   ✅ Guardado: Hotel Del Valle Azapa
   ✅ Guardado: Hotel & Spa Las Taguas
   ✅ Guardado: Le Prince Arica
   ✅ Guardado: Raymi House Hostel
   ✅ Guardado: Hostel Posada de Gallo
   ✅ Guardado: Antay Hotel & Spa
   ✅ Guardado: Hotel Savona
   ✅ Guardado: Hotel Puerto Chinchorro
   ✅ Guardado: Hostel Willka Kuti Backpackers
   ✅ Guardado: Hostal D' Silvia
   ✅ Guardado: Diego De Almagro Arica
   ✅ Guardado: Novotel Arica
   ✅ Guardado: Apart Hotel Viscachani
   ✅ Guardado: Hotel Apacheta
   ✅ Guardado: Hostal Copa Arica
   ✅ Guardado: Hotel Qamiri
   ✅ Guardado: Illariy Wasi Hostal
   ✅ Guardado: Casa Huespedes Angoba
  